**MOUNTING GOOGLE DRIVE**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**IMPORTING PACKAGES**

In [2]:
import os                     # used for interacting with the file system..
import PIL                    # PIL is the Python Imaging Library adds image processing capabilities..
from PIL import Image         # Image module is used to load images from files..
import torch                  # Torch is an open source machine learning library used for developing and training neural network based deep learning models..
import torch.nn as nn         # Neural networks can be constructed using the torch.nn package..
import torch.optim as optim   # optim package to define an Optimizer that will update the weights for us..
from torch.utils.data import Dataset, DataLoader   # used for loading data..
from torch.autograd import Variable   # Variables are just wrappers for the tensors..
# from time import time
import torchvision            #  torchvision package consists of popular datasets, model architectures, and common image transformations for computer vision.
import torchvision.transforms as transforms     # Transforms are common image transformations.
from torchsummary import summary
import itertools
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

**CHANGING CURRENT WORKING DIRECTORY**

In [3]:
os.chdir(os.getcwd()+"/drive/MyDrive")
path=os.getcwd()
print(path)

/content/drive/MyDrive


**CREATING CLASS FOR LOADING DATA**

In [4]:
dataset='dataset/data'
classes=['COVID','Non-COVID']
class CTDataset(Dataset):
    def __init__(self,path,dataset,classes,transform=None):
        self.path=path
        self.dataset=dataset
        self.classes=classes
        self.num_cls=len(self.classes)
        self.img_list=[]
        for c in range(self.num_cls):
            cls_list=[[os.path.join(self.path,self.dataset,self.classes[c],item),c] for item in os.listdir(os.path.join(self.path,self.dataset,self.classes[c]))]
            self.img_list+=cls_list
        self.transform=transform
    def __len__(self):
        return len(self.img_list)
    
    
    def __getitem__(self,idx):
        img_path=self.img_list[idx][0]
        image=Image.open(img_path).convert('RGB')
        if self.transform:
            image=self.transform(image)
        sample={'img':image,
               'label':int(self.img_list[idx][1])}
        return sample
        

**UNTRANSFORMED DATA**

In [5]:
training_set_untransformed=CTDataset(path,dataset,classes)
print(training_set_untransformed.__len__())

3225


**CREATING TRANSFORMED DATA**

In [6]:
transform_train = transforms.Compose([transforms.Resize((224,224)),transforms.RandomApply([
        torchvision.transforms.RandomRotation(10),
        transforms.RandomHorizontalFlip()],0.7),
		transforms.ToTensor()])

new_created_images=[]
for data in training_set_untransformed:
    if data['label']==0:
        for k in range(5):
            transformed_image = transform_train(data['img'])
            new_created_images.append((transformed_image,0))
    else:
      for k in range(3):
        transformed_image = transform_train(data['img'])
        new_created_images.append((transformed_image,1))
print(len(new_created_images))

12875


**SPLITTING THE DATA**

In [7]:
train_size = int(0.8 * len(new_created_images))
validation_size = len(new_created_images) - train_size
train_dataset, validation_dataset = torch.utils.data.random_split(new_created_images, [train_size,validation_size])

**LOADING THE TRAINING DATASET**

In [8]:
training_generator = DataLoader(train_dataset,shuffle=True,batch_size=32,pin_memory=True)

**CONNECTING GPU DEVICE**

In [9]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")
torch.cuda.empty_cache()
print(device)

cuda:0


**INSTALLING EFFICIENTNET**

In [10]:
pip install efficientnet_pytorch

  Created wheel for efficientnet-pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16446 sha256=d9c550bbe7691eb77a22f3afc3191f2177037d811a43eb17053ddeaadda7638d
  Stored in directory: /root/.cache/pip/wheels/0e/cc/b2/49e74588263573ff778da58cc99b9c6349b496636a7e165be6
Successfully built efficientnet-pytorch


**LOADING PRETRAINED MODEL EFFICIENTNET-B0**

In [11]:
from efficientnet_pytorch import EfficientNet
model = EfficientNet.from_pretrained('efficientnet-b0', num_classes=2)

Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth


  0%|          | 0.00/20.4M [00:00<?, ?B/s]

Loaded pretrained weights for efficientnet-b0


**SAVING MODEL ON GPU DEVICE**

In [12]:
model.to(device)
print(summary(model, input_size=(3, 224, 224)))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
         ZeroPad2d-1          [-1, 3, 225, 225]               0
Conv2dStaticSamePadding-2         [-1, 32, 112, 112]             864
       BatchNorm2d-3         [-1, 32, 112, 112]              64
MemoryEfficientSwish-4         [-1, 32, 112, 112]               0
         ZeroPad2d-5         [-1, 32, 114, 114]               0
Conv2dStaticSamePadding-6         [-1, 32, 112, 112]             288
       BatchNorm2d-7         [-1, 32, 112, 112]              64
MemoryEfficientSwish-8         [-1, 32, 112, 112]               0
          Identity-9             [-1, 32, 1, 1]               0
Conv2dStaticSamePadding-10              [-1, 8, 1, 1]             264
MemoryEfficientSwish-11              [-1, 8, 1, 1]               0
         Identity-12              [-1, 8, 1, 1]               0
Conv2dStaticSamePadding-13             [-1, 32, 1, 1]             288
         I

**TRAINING THE MODEL**

In [ ]:
history_accuracy=[]
history_loss=[]
epochs = 10
history_accuracy1=[]
history_loss1=[]

criterion = nn.CrossEntropyLoss()
lr_decay=0.99
learning_rate=1e-4
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
eye = torch.eye(2).to(device)

for epoch in range(epochs):
    running_loss = 0.0
    running_loss1 = 0.0
    correct=0
    total=0
    class_correct = list(0. for _ in classes)
    class_total = list(0. for _ in classes)
    
    for i, data in enumerate(training_generator, 0):
        inputs, labels = data
        #t0 = time()
        inputs, labels = inputs.to(device), labels.to(device)
        labels = eye[labels]
        optimizer.zero_grad()
        torch.cuda.empty_cache()
        outputs = model(inputs)
        loss = criterion(outputs, torch.max(labels, 1)[1])
        _, predicted = torch.max(outputs, 1)
        _, labels = torch.max(labels, 1)
        c = (predicted == labels.data).squeeze()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        accuracy = float(correct) / float(total)
        
        history_accuracy.append(accuracy)
        history_loss.append(loss)
        
        loss.backward()
        optimizer.step()
        
        for j in range(labels.size(0)):
            label = labels[j]
            class_correct[label] += c[j].item()
            class_total[label] += 1
        
        running_loss += loss.item()
        running_loss1+=loss.item()*inputs.size(0)
       #print( "Epoch : ",epoch+1," Batch : ", i+1," Loss :  ",running_loss/(i+1)," Accuracy : ",accuracy,"Time ",round(time()-t0, 2),"s" )
    history_loss1.append(running_loss1/ len(training_generator.dataset))
    history_accuracy1.append(accuracy)

    for k in range(len(classes)):
        if(class_total[k]!=0):
            print('Accuracy of %5s : %2d %%' % (classes[k], 100 * class_correct[k] / class_total[k]))
        
    print('[%d epoch] Accuracy of the network on the Training images: %d %%' % (epoch+1, 100 * correct / total))

Accuracy of COVID : 90 %
Accuracy of Non-COVID : 85 %
[1 epoch] Accuracy of the network on the Training images: 88 %


**LOSS AND ACCURACY GRAPH**

In [ ]:
'''plt.plot(history_accuracy)
plt.plot(history_loss)'''
plt.plot(history_accuracy, label='Training accuracy')
plt.plot(history_loss, label='Training loss')
plt.grid()
plt.title('Training Loss and accuracy')
plt.xlabel('Batches')
plt.ylabel("Accuracy & Loss")
plt.legend()
#plt.savefig('efficient.eps', format='eps')
plt.savefig('efficientnet_graph.jpg', format='jpg', dpi=1200)
print(len(history_loss1))

**VALIDATING THE MODEL**

In [ ]:
model.eval()
with torch.no_grad():
  correct_counter=0
  predicted=[]
  t_labels=[]
  for i in range(len(validation_dataset)):
      image_tensor = validation_dataset[i][0].unsqueeze_(0)
      input = Variable(image_tensor)
      input = input.to(device)
      output = model(input)
      index = output.data.cpu().numpy().argmax()
      predicted.append(index)
      t_labels.append(validation_dataset[i][1])

**CONFUSION MATRIX**

In [ ]:
cm = confusion_matrix(t_labels, predicted)

def plot_confusion_matrix(cm,
                          classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
  
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)

    # Specify the tick marks and axis text
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=90)
    plt.yticks(tick_marks, classes)

    # The data formatting
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.

    # Print the text of the matrix, adjusting text colour for display
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()
plot_confusion_matrix(cm, classes)
print('ACCURACY:',(cm[0][0]+cm[1][1])/cm.sum())
print('Specificity:',(cm[1][1])/(cm[1][1]+cm[1][0]))
print('Sensitivity:',cm[0][0]/(cm[0][0]+cm[0][1]))
print('Precision:',cm[0][0]/(cm[0][0]+cm[1][0]))

In [ ]:
test_transforms = transforms.Compose([transforms.Resize((224,224)),transforms.RandomApply([
        torchvision.transforms.RandomRotation(10),
        transforms.RandomHorizontalFlip()],0.7),
                                      transforms.ToTensor(),
                                     ])

def predict_image(image):
    image_tensor = test_transforms(image)
    image_tensor = image_tensor.unsqueeze_(0)
    input = Variable(image_tensor)
    input = input.to(device)
    output = model(input)
    
    index = output.data.cpu().numpy().argmax()
    return index

**TESTING THE MODEL WITH SAMPLE IMAGE**

In [ ]:
image1=Image.open('/content/drive/My Drive/dataset/data/Non-Covid.png').convert('RGB')
print(classes[predict_image(image1)])